In [ ]:
import numpy as np


from pathlib import Path

import prism

from pathlib import Path

import prism

from imagematerials.factory import ModelFactory, Sector
from imagematerials.model import (
    GenericMaterials,
    GenericStocks,
    MaterialIntensities,
    RestOf, 
    SharesInflowStocks
)
from imagematerials.preprocessing import get_preprocessing_data

In [ ]:
scenario_list = {
    "SSP2_baseline":("SSP2_baseline", None),
                 }

In [ ]:
base_path = Path("..", "data", "raw")
climate_scenarios = list(scenario_list.keys())

In [ ]:
model_runs = {}

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {scen_id}")
    print(f"Circular economy scenario: {circular_scen}")
    
    # needed for electricity
    scen = climate_scen.split("_")[0]
    variant = "_".join(climate_scen.split("_")[1:])
    
    climate_policy_scenario_dir = base_path / Path("image", climate_scen)
    if circular_scen is not None:
        circular_economy_scenario_dirs = {
                scenario: base_path / Path("circular_economy_scenarios", scenario) for scenario in circular_scen
            }
    else:
        circular_economy_scenario_dirs = None

    # Define the complete timeline, including historic tail
    time_start = 1971
    complete_timeline = prism.Timeline(time_start, 2100, 1)
    simulation_timeline = prism.Timeline(1971, 2100, 1)

    # BUILDINGS
    bld_sector = get_preprocessing_data("buildings", base_path, 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs) 
    # VEHICLES
    vhc_sector = get_preprocessing_data("vehicles", base_path, 
                                        climate_policy_scenario_dir, 
                                        circular_economy_scenario_dirs)
    prep_data_vhc = vhc_sector.prep_data
    vhc_sector = Sector('vehicles', prep_data_vhc)
    
    # #ELECTRICITY
    elc_sector = get_preprocessing_data("electricity", base_path, climate_policy_scenario_dir, circular_economy_scenario_dirs, cache = False)
    elc_gen, elc_grid_lines, elc_grid_add, elc_stor_phs, elc_stor_other = (
    {sec.name: sec for sec in elc_sector}[k]
    for k in ["elc_gen", "elc_grid_lines", "elc_grid_add", "elc_stor_phs", "elc_stor_other"])

    # REST OF SECTOR
    rest_sector = get_preprocessing_data("rest_of", 
                                         base_path,
                                         climate_policy_scenario_dir,
                                         circular_economy_scenario_dirs,
                                         climate_scen)
    
    factory = ModelFactory(
    [bld_sector, vhc_sector, elc_gen, rest_sector,  # 
     elc_grid_lines, elc_grid_add, elc_stor_phs, elc_stor_other], complete_timeline
    ).add(GenericStocks, ["buildings", "vehicles", "elc_gen", "elc_grid_lines", "elc_grid_add", 'elc_stor_phs'] 
    ).add(GenericMaterials,  "vehicles"
    ).add(SharesInflowStocks, "elc_stor_other"
    ).add(MaterialIntensities, ["buildings", "elc_gen", "elc_grid_lines", "elc_grid_add", 
                                "elc_stor_phs", "elc_stor_other"] 
    ).add(RestOf, "rest_of"
    )
    model = factory.finish()
    model_runs[climate_scen] = model

  
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline)